In [1]:
# Conversational Context: ....
# Welcome! In this notebook, you’ll learn ....

In [2]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# (optional) Verify installation
%pip show google-adk

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [5]:
# Import Required Modules
import requests
from typing import Dict, Any, Optional
from datetime import datetime, timedelta
from collections import Counter
from dateutil import parser as date_parser
import calendar
import re

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.genai import types
from IPython.display import Markdown, display

In [6]:
# OpenWeather API: Set Up
OPEN_WEATHER_API_KEY = os.environ.get("OPEN_WEATHER_API_KEY")
if not OPEN_WEATHER_API_KEY:
    raise ValueError("OPEN_WEATHER_API_KEY environment variable not set.  Please set it.")

In [7]:
# Helper Functions: Parse flexible date expressions for weather queries
def parse_flexible_date_range(date_str: str) -> Optional[tuple]:
    """
    Parses a wide variety of date expressions and returns (start_date, end_date).
    Supports:
    - 'today', 'tomorrow', 'in 3 days'
    - 'this weekend', 'next weekend'
    - 'next week', 'this week'
    - 'first week of July', 'second week of August', etc.
    - Specific dates: '2025-07-01', 'July 1', '07-01'
    Returns (datetime, datetime) or None if parsing fails.
    """
    date_str = date_str.lower().strip()
    today = datetime.today()
    weekday = today.weekday()  # Monday=0, Sunday=6

    # Today, tomorrow, in X days
    target_date = None
    if date_str in ["today", "now"]:
        target_date = today
    elif date_str == "tomorrow":
        target_date = today + timedelta(days=1)
    
    match = re.match(r"in (\\d+) days?", date_str)
    if match:
        days = int(match.group(1))
        target_date = today + timedelta(days=days)

    if target_date:
        start_of_day = target_date.replace(hour=0, minute=0, second=0, microsecond=0)
        end_of_day = target_date.replace(hour=23, minute=59, second=59, microsecond=999999)
        return start_of_day, end_of_day
    
    match = re.match(r"(?:next|for the next) (\d+) days?", date_str)
    if match:
        days = int(match.group(1))
        start = today
        end = today + timedelta(days=days - 1)
        return start, end

    # This week (Monday to Sunday)
    if date_str == "this week":
        start = today - timedelta(days=weekday)
        end = start + timedelta(days=6)
        return start, end

    # Next week (Monday to Sunday)
    if date_str == "next week":
        start = today - timedelta(days=weekday) + timedelta(days=7)
        end = start + timedelta(days=6)
        return start, end

    # This weekend (Saturday & Sunday)
    if date_str == "this weekend":
        saturday = today + timedelta((5 - weekday) % 7)
        sunday = saturday + timedelta(days=1)
        return saturday, sunday

    # Next weekend (Saturday & Sunday)
    if date_str == "next weekend":
        saturday = today + timedelta((5 - weekday) % 7 + 7)
        sunday = saturday + timedelta(days=1)
        return saturday, sunday

    # "first week of July", "second week of August", etc.
    match = re.match(r"(first|second|third|fourth|last) week of (\w+)", date_str)
    if match:
        week_map = {
            "first": 0, "second": 1, "third": 2, "fourth": 3, "last": -1
        }
        week_num = week_map[match.group(1)]
        month_str = match.group(2)
        try:
            month = list(calendar.month_name).index(month_str.capitalize())
            year = today.year
            # If the month has already passed, assume next year
            if month < today.month:
                year += 1
            cal = calendar.monthcalendar(year, month)
            if week_num == -1:
                week = cal[-1]
            else:
                week = cal[week_num]
            # Get Monday (0) and Sunday (6) of that week
            start_day = week[0] if week[0] != 0 else week[calendar.SATURDAY]
            end_day = week[6] if week[6] != 0 else week[calendar.FRIDAY]
            start = datetime(year, month, start_day)
            # Find first non-zero day for start, last non-zero for end
            for d in week:
                if d != 0:
                    start = datetime(year, month, d)
                    break
            for d in reversed(week):
                if d != 0:
                    end = datetime(year, month, d)
                    break
            return start, end
        except Exception:
            return None

    # Try parsing as a specific date or date range
    try:
        # "July 1", "07-01", "2025-07-01"
        dt = date_parser.parse(date_str, fuzzy=True, default=today)
        return dt, dt
    except Exception:
        pass

    # "July 1 to July 5", "07-01 to 07-05"
    if " to " in date_str:
        parts = date_str.split(" to ")
        try:
            start = date_parser.parse(parts[0], fuzzy=True, default=today)
            end = date_parser.parse(parts[1], fuzzy=True, default=today)
            return start, end
        except Exception:
            return None

    return None


In [17]:
# Custom Tool: Get Current weather (stateful version)
def get_current_weather_from_openweather_stateful(city: str, tool_context) -> Dict[str, Any]:
    preferred_unit = tool_context.state.get("user_preference_temperature_unit", "metric")
    unit_symbol = "°C" if preferred_unit == "metric" else "°F"
    if not city:
        city = tool_context.state.get("last_city_checked_stateful")
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPEN_WEATHER_API_KEY}&units={preferred_unit}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        if data["cod"] == 200:
            description = data["weather"][0]["description"]
            temperature = data["main"]["temp"]
            humidity = data["main"]["humidity"]
            wind_speed = data["wind"]["speed"]
            report = (
                f"The weather in {city} is {description} with a temperature of {temperature}{unit_symbol}. "
                f"Humidity is {humidity}% and wind speed is {wind_speed} m/s."
            )
            tool_context.state["last_city_checked_stateful"] = city
            return {"status": "success", "report": report}
        else:
            return {"status": "error", "error_message": f"OpenWeatherMap API Error: {data['message']}"}
    except requests.exceptions.RequestException as e:
        return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    except KeyError as e:
        return {"status": "error", "error_message": f"Error parsing weather data: Missing key: {e}"}
    except Exception as e:
        return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}

In [31]:
# Custom Tool: Get weather summary for a flexible date range (stateful version)
def get_weather_summary_from_openweather_stateful(
    city: Optional[str], date_expr: Optional[str], tool_context
) -> Dict[str, Any]:
    if not city:
        city = tool_context.state.get("last_city_checked_stateful")
    if not city:
        return {"status": "error", "error_message": "No city specified or remembered."}
    if not date_expr:
        date_expr = tool_context.state.get("last_date_expr_stateful")
    if not date_expr:
        return {"status": "error", "error_message": "No date or date range specified or remembered."}

    preferred_unit = tool_context.state.get("user_preference_temperature_unit", "metric")
    unit_symbol = "°C" if preferred_unit == "metric" else "°F"
    date_range = parse_flexible_date_range(date_expr)
    if not date_range:
        return {"status": "error", "error_message": f"Could not parse date expression: '{date_expr}'"}
    start, end = date_range
    num_days = (end - start).days + 1

    print(start, end, num_days)  # Debugging line to check parsed date range

    try:
        url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&appid={OPEN_WEATHER_API_KEY}&units={preferred_unit}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        current_url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPEN_WEATHER_API_KEY}&units={preferred_unit}"
        current_response = requests.get(current_url)
        current_response.raise_for_status()
        current_data = current_response.json()
        all_temps = []
        all_descriptions = []
        for entry in data.get("list", []):
            entry_date = datetime.fromtimestamp(entry["dt"])
            if start <= entry_date <= end:
                all_temps.append(entry["main"]["temp"])
                all_descriptions.append(entry["weather"][0]["description"])
        if not all_temps:
            all_temps = [current_data["main"]["temp"]]
            all_descriptions = [current_data["weather"][0]["description"]]
        print(all_temps)  # Debugging line to check collected temperatures
        min_temp = round(min(all_temps), 1)
        max_temp = round(max(all_temps), 1)
        avg_temp = round(sum(all_temps) / len(all_temps), 1)
        weather_types = [desc for desc, _ in Counter(all_descriptions).most_common(3)]
        weather_summary = {
            "destination": data["city"]["name"] if "city" in data else city,
            "country": data["city"]["country"] if "city" in data else "",
            "trip_length": num_days,
            "date_range": f"{start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}",
            "avg_temp": avg_temp,
            "temp_range": f"{min_temp}{unit_symbol} to {max_temp}{unit_symbol}",
            "weather_types": weather_types
        }
        tool_context.state["last_city_checked_stateful"] = city
        tool_context.state["last_date_expr_stateful"] = date_expr
        return {"status": "success", "summary": weather_summary}
    except requests.exceptions.RequestException as e:
        return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    except KeyError as e:
        return {"status": "error", "error_message": f"Error parsing weather data: Missing key: {e}"}
    except Exception as e:
        return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}


    #     # Update both session state and memory
    #     tool_context.state["last_city_checked_stateful"] = city
    #     tool_context.memory["last_city_checked_memory"] = city
    #     tool_context.state["last_date_expr_stateful"] = date_expr
    #     tool_context.memory["last_date_expr_memory"] = date_expr

    #     return {"status": "success", "summary": weather_summary}

    # except requests.exceptions.RequestException as e:
    #     return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    # except KeyError as e:
    #     return {"status": "error", "error_message": f"Error parsing weather data: Missing key: {e}"}
    # except Exception as e:
    #     return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}

In [44]:
# Weather Agent: Definition
weather_agent = Agent(
    name="weather_agent_stateful",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Provides current weather and weather summaries, using user preferences and context.",
    instruction=(
        "You are a helpful weather assistant. "
        "If the user does not specify a city, use the last city checked from session state. "
        "If the user does not specify a date, use the last date range from session state. "
        "Accept natural language date expressions like 'this weekend' or 'next week'. "
        "When the user asks for the current weather in a city, "
        "use the 'get_current_weather_from_openweather_stateful' tool. "
        "When the user asks for weather for a date or date range, "
        "use the 'get_weather_summary_from_openweather_stateful' tool. "
        "If a tool returns an error, inform the user politely. "
    ),
    tools=[get_current_weather_from_openweather_stateful, get_weather_summary_from_openweather_stateful],
)

In [45]:
# Set up the session and user input
APP_NAME = "wanderwise_app"
USER_ID = "user_001"
SESSION_ID = "weather_demo_session"

session_service = InMemorySessionService()
initial_state = {
    "user_preference_temperature_unit": "imperial"
}

# Create session using the correct method signature
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state=initial_state
)

memory_service = InMemoryMemoryService()

runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service
)

In [46]:
# Helper function to run and display agent responses
async def ask_weather_agent(runner, prompt):
    content = types.Content(role="user", parts=[types.Part(text=prompt)])
    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            # Extract only the text parts
            text_parts = [part.text for part in event.content.parts if hasattr(part, 'text') and part.text]
            # Join all text parts (in case there are multiple)
            text_response = "\n".join(text_parts)
            display(Markdown(f"**Prompt:** {prompt}\n\n---\n\n**Agent Response:**\n\n{text_response}"))


In [47]:
# Helper Function: To show session state
async def show_context():
    # Get the latest session state
    current_session = await session_service.get_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
    print("Session state (per conversation):")
    for k, v in current_session.state.items():
        print(f"  {k}: {v}")

In [48]:
# Helper function to update session state properly
async def update_temperature_unit(new_unit):
    """Update the temperature unit preference in the session state"""
    current_session = session_service.sessions[APP_NAME][USER_ID][SESSION_ID]
    current_session.state["user_preference_temperature_unit"] = new_unit
    print(f"Updated temperature unit to: {new_unit}")

In [49]:
# Example Interactions

# NOTE: OpenWeatherMap free tier only provides 5 days of forecast. If you ask for more, it will use the current weather as a fallback.

# Ask for current weather (uses state for unit, updates memory with last city)
await ask_weather_agent(runner, "What's the weather in San Francisco?")
await show_context()

# Ask for a weather summary for a date range (uses state/memory for city and date)
await ask_weather_agent(runner, "And, what will the weather be like for the next 3 days?")  # Should use last city
await show_context()

# Ask for a weather summary for a different city and flexible date
await ask_weather_agent(runner, "What's the weather in Paris tomorrow?")
await show_context()

await update_temperature_unit("metric") 
# Ask for a weather summary for a different city and flexible date
await ask_weather_agent(runner, "What's the weather in Paris tomorrow?")
await show_context()

await ask_weather_agent(runner, "And for the same date range, what is the weather in London?")
await show_context()

# Ask for the last city checked (uses memory)
await ask_weather_agent(runner, "What was the last city I checked the weather for?")
await show_context()


**Prompt:** What's the weather in San Francisco?

---

**Agent Response:**

The weather in San Francisco is few clouds with a temperature of 57.96°F. Humidity is 76% and wind speed is 18.41 m/s.


Session state (per conversation):
  user_preference_temperature_unit: imperial
  last_city_checked_stateful: San Francisco


2025-06-02 01:26:40.494843 2025-06-04 01:26:40.494843 3
[58.78, 55.51, 51.55, 50.83, 55.53, 70.21, 61.97, 59.52, 56.64, 55.06, 54.21, 53.74, 56.1, 61.95, 62.51, 59.97]


**Prompt:** And, what will the weather be like for the next 3 days?

---

**Agent Response:**

OK. For the next 3 days in San Francisco, the weather will be a mix of clear sky, few clouds, and scattered clouds. The temperature will range from 50.8°F to 70.2°F, with an average temperature of 57.8°F.


Session state (per conversation):
  user_preference_temperature_unit: imperial
  last_city_checked_stateful: San Francisco
  last_date_expr_stateful: next 3 days


...

2025-06-03 00:00:00 2025-06-03 23:59:59.999999 1
[11.63, 11.22, 13.26, 14.52, 13.4, 19.9, 17.69, 14.21]


**Prompt:** And for the same date range, what is the weather in London?

---

**Agent Response:**

The weather in London tomorrow will be a mix of overcast clouds, light rain, and scattered clouds. The temperature will range from 11.2°C to 19.9°C, with an average temperature of 14.5°C.


Session state (per conversation):
  user_preference_temperature_unit: metric
  last_city_checked_stateful: London
  last_date_expr_stateful: tomorrow


**Prompt:** What was the last city I checked the weather for?

---

**Agent Response:**

The last city you checked the weather for was London.


Session state (per conversation):
  user_preference_temperature_unit: metric
  last_city_checked_stateful: London
  last_date_expr_stateful: tomorrow


In [16]:
# 🎉 Congratulations!
# You now have a ......